## CTE'S (Common Table Expression) is simply a temporary, disposable table.
##### Imagine you need to filter your data using a Window Function, but SQL has a strict rule: you cannot use a WHERE clause on a Window Function directly.
#### A CTE is the workaround. It allows you to:
##### 1. Build a temporary table that includes your Window Function results.
##### 2. Run a normal SELECT ... WHERE query on that temporary table.
##### 3. Once the query finishes, the temporary table instantly disappears.
##### It is like using a piece of scratch paper to solve the first half of a math problem before writing down the final answer.

### Question: 
#### The Scenario: Cleaning the Bank Data
##### 1. A glitch in the bank's system accidentally charged Alice for her $50 coffee three times on the exact same day.
##### Bob and Charlie's transactions are completely fine.
##### 2. Your manager wants a clean report that only shows valid transactions. You need to filter out the duplicates so that Alice only has one $50 charge.

### STEPS to fallow:
##### Step 1 (The CTE): Build a temporary table. Use ROW_NUMBER() to assign a number to every row.
##### Hint: You must use PARTITION BY Account, Date, Amount so the database groups the identical coffee charges into the exact same bucket.
##### Step 2 (The Final Query): Select all columns from your temporary table, but strictly filter it to only show rows where the row number is 1. This will give you the completely clean dataset!

In [4]:
import pandas as pd

# The database table with the accidental duplicates
df_bank = pd.DataFrame({
    'Account': ['Alice', 'Alice', 'Alice', 'Bob', 'Charlie'],
    'Date': ['2023-10-01', '2023-10-01', '2023-10-01', '2023-10-01', '2023-10-02'],
    'Amount': [50, 50, 50, 100, 75]
})

df_bank

,Account,Date,Amount
0,Alice,2023-10-01,50
1,Alice,2023-10-01,50
2,Alice,2023-10-01,50
3,Bob,2023-10-01,100
4,Charlie,2023-10-02,75


In [ ]:
import duckdb
query_bank = """
WITH Ranked_Transactions AS(
    SELECT
        Account,
        Date,
        Amount,
        ROW_NUMBER() OVER(PARTITION BY Account ORDER BY Date) AS Transactions
    FROM df_bank
)

SELECT
    Account,
    Date,
    Amount
FROM Ranked_Transactions
WHERE Transactions = 1;
"""
duckdb.query(query_bank).df()

,Account,Date,Amount
0,Alice,2023-10-01,50
1,Bob,2023-10-01,100
2,Charlie,2023-10-02,75


## Question
##### The CEO wants to give a special bonus to the Top 2 highest-selling employees in every single region.
##### You have a list of sales data from the North and South regions.

### STEPS 
##### Step 1 (CTE): Use DENSE_RANK() to rank employees by their sales from highest to lowest. You must use PARTITION BY Region so the North and South teams don't compete against each other—they each need their own Rank 1 and Rank 2!
##### Step 2 (Final Query): Filter your temporary table to only show employees where their rank is 1 or 2 (Hint: WHERE Rank_Number <= 2).

In [8]:
import pandas as pd

df_sales = pd.DataFrame({
    'EmpName': ['Emma', 'Liam', 'Olivia', 'Noah', 'Ava', 'William'],
    'Region': ['North', 'North', 'North', 'South', 'South', 'South'],
    'Sales': [15000, 12000, 12000, 20000, 18000, 10000]
})

df_sales

,EmpName,Region,Sales
0,Emma,North,15000
1,Liam,North,12000
2,Olivia,North,12000
3,Noah,South,20000
4,Ava,South,18000
5,William,South,10000


In [11]:
import duckdb
query_sales = """
WITH Ranking_Sales AS(
    SELECT
        EmpName,
        Region,
        Sales,
        DENSE_RANK() OVER(PARTITION BY Region ORDER BY Sales DESC) AS Region_Ranking
    FROM df_sales
)

SELECT
    EmpName,
    Region,
    Sales,
    Region_Ranking
FROM Ranking_Sales
WHERE Region_Ranking <=2;
"""

duckdb.query(query_sales).df()

,EmpName,Region,Sales,Region_Ranking
0,Noah,South,20000,1
1,Ava,South,18000,2
2,Emma,North,15000,1
3,Liam,North,12000,2
4,Olivia,North,12000,2
